In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math
import random

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 81.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=b4e2d38195ba708791b89749ce2c64147203e882e148c022b4501595a158dd4a
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# Qubit symbols for the two bases:
#   Standard basis ('s'):  0 -> |0>,  1 -> |1>
#   Diagonal basis ('d'):  0 -> |+>,  1 -> |->

QUBIT_SYMBOL = {
    (0, 's'): '0',
    (1, 's'): '1',
    (0, 'd'): '+',
    (1, 'd'): '-',
}

def encode(bit, basis):
    """
    Alice encodes a bit into a qubit using the chosen basis.
      bit=0, basis='s' : |0>  (no gates)
      bit=1, basis='s' : |1>  (X gate)
      bit=0, basis='d' : |+>  (H gate)
      bit=1, basis='d' : |->  (X then H)
    """
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == 'd':
        qc.h(0)
    return qc

def measure(circuit, basis):
    """
    Measure a qubit in the chosen basis.
    Standard basis : measure directly.
    Diagonal basis : apply H first (converts |+>->|0>, |->->|1>), then measure.
    """
    qc = circuit.copy()
    if basis == 'd':
        qc.h(0)
    qc.measure(0, 0)
    return qc

def run_one(circuit):
    """Simulate a single-shot measurement and return the bit result (int)."""
    backend = BasicSimulator()
    t_qc    = transpile(circuit, backend)
    counts  = backend.run(t_qc, shots=1).result().get_counts()
    return int(list(counts.keys())[0])

# Number of qubits Alice sends
N = 20

# Alice randomly generates bits and bases
alice_bits  = [random.randint(0, 1) for _ in range(N)]
alice_bases = [random.choice(['s', 'd']) for _ in range(N)]

# Alice encodes each bit into a qubit circuit
alice_circuits = [encode(b, bas) for b, bas in zip(alice_bits, alice_bases)]

# Bob randomly chooses a basis for each qubit and measures
bob_bases = [random.choice(['s', 'd']) for _ in range(N)]
bob_bits  = [run_one(measure(qc, b)) for qc, b in zip(alice_circuits, bob_bases)]

# Alice and Bob compare bases publicly and keep only matching positions
matching_pos = [i for i in range(N) if alice_bases[i] == bob_bases[i]]
alice_key    = [alice_bits[i] for i in matching_pos]
bob_key      = [bob_bits[i]   for i in matching_pos]

In [3]:
# Print the BB84 table

def print_row(label, values):
    print(f"{label:<12}", end=" ")
    for v in values:
        print(f"{str(v):<5}", end="")
    print()

print("BB84 Key Distribution (No Attacker)")
print("=" * 108)
print_row("Index:",   list(range(N)))
print_row("A bit:",   alice_bits)
print_row("A basis:", alice_bases)
print_row("qubit:",   [QUBIT_SYMBOL[(alice_bits[i], alice_bases[i])] for i in range(N)])
print_row("B basis:", bob_bases)
print_row("B bit:",   [bob_bits[i] if alice_bases[i] == bob_bases[i] else '?' for i in range(N)])
print_row("match:",   ['<<' if alice_bases[i] == bob_bases[i] else '' for i in range(N)])
print("=" * 108)
print()
print(f"Matching positions : {matching_pos}")
print(f"Alice's key        : {alice_key}")
print(f"Bob's key          : {bob_key}")
print(f"Keys match         : {alice_key == bob_key}")
print(f"Key length         : {len(alice_key)} / {N} bits")

BB84 Key Distribution (No Attacker)
Index:       0    1    2    3    4    5    6    7    8    9    10   11   12   13   14   15   16   17   18   19   
A bit:       0    0    0    0    0    1    1    0    0    0    0    0    1    1    0    0    1    0    0    0    
A basis:     s    d    d    s    s    d    s    d    s    s    d    s    d    d    s    s    d    s    s    s    
qubit:       0    +    +    0    0    -    1    +    0    0    +    0    -    -    0    0    -    0    0    0    
B basis:     s    d    s    s    d    s    s    s    d    s    d    d    s    s    s    s    d    d    d    d    
B bit:       0    0    ?    0    ?    ?    1    ?    ?    0    0    ?    ?    ?    0    0    1    ?    ?    ?    
match:       <<   <<        <<             <<             <<   <<                  <<   <<   <<                  

Matching positions : [0, 1, 3, 6, 9, 10, 14, 15, 16]
Alice's key        : [0, 0, 0, 1, 0, 0, 0, 0, 1]
Bob's key          : [0, 0, 0, 1, 0, 0, 0, 0, 1]
Keys match    